# 1. Введение

## 1.1 Описание задачи

  В данной работе решена задача детекции Компьютерного Зрения (далее CV - Computer Vision): выделение рамок (далее BB - Bounding Box) вокруг шахматных фигур и их классификация.
  
  Ракурс для изображение для инференса выбран строго перпендикулярно центру доски. Количество фигур не превышало 12 (6 уникальных по форме фигур черного и белого цвета). Все фигуры стоят строго в рамках доски. Источники освещения были выбраны так, чтобы сами фигуры не отбрасывали тени.

##1.2 Описание датасета

Конечный датасет представляет собой 65 000 фотографий с размеченными BB.

15 000 фотографий датасета - синтетические данные. На каждой из них представлено ровно 12 уникальных по форме и цвету фигур с искусственным освещением без теней от фигур, стоящих на шахматной доске.

50 000 фотографий общего датасета это реальные снимки шахматных фигур, по одной на снимок. Каждая фигура в снимке находится ровно в центре и стоит под наклоном.


Пример синтетического датасета:

![2.png](../docs/assets/synthetic_blender_scene.png)

Пример реального датасета:

![rook_white_0104_black_angle_15.png](../docs/assets/yolo_board_detection_real.png)

## 1.3 Выбранные методы CV

Для решения задачи сегментации были выбраны 3 метода: YOLOv11n, SSD и FasterRCNN.

Методы были выбраны по следующим причинам:
* Лучшие модели детекции с существующими backbone
* По времени обучения: YOLOv11n < SSD < FasterRCNN
* По качеству обучения: YOLOv11n < SSD < FasterRCNN

# 2. Датасет

## 2.1 Искусственные данные

### 2.1.1 Описание

#### Откуда были взяты фигуры и какая программа использовалась для создания

Фигуры были взяты как 3D модели с сайта Cults3D:

[Chess by Lubal](https://cults3d.com/ru/3d-model/igra/ajedrez-lubal-2)

\

Для создания датасета использовался Blender (программа для 3D моделлирования) со встроенной функцией исполнения кода Python.

#### Сколько фигур на доске, как они расположены и почему

На каждой фотографии на доске ровно 12 уникальных по цвету и геометрии фигур.

Каждая фигура имеет случайный поворт от 0 до 360 градусов вокруг своей оси.

Каждая фигура находится в рамках клетки доски. Фигуры не пересекаются и могут занимать любую клетку доски. Такой выбор обусловлен дальнейшей простотой разметки.\
Также, располагая фигуры разного цвета на черных и белых клетках в любых клетках доски, получаются разнообразные данные, что увеличивает обобщающую способность моделей CV.

![12.png](../docs/assets/synthetic_pieces_distribution.png)

![image.png](../docs/assets/synthetic_multi_piece_scene.png)

#### Размер конечной фотографии и BB

Размер изображений был выбран 500x500.

Размер клеток постоянный и одинаковый, поэтому каждый BB был выбран как 70x70 вокруг центра кажой клетки с фигурой в ней.

#### Разметка

Разметка для синтетического датасета происходила автоматически:

При создании датасета был сформирован CSV файл со следующими данными:
* название фотографии
* класс фигруы (от 0 до 11)
* координаты клетки доски (количество клеток от угла доски по x и по y)

Используя эти данные были определены координаты центров фигур на изображении.

Доска на каждом изображении находилась в одном и том же положении, поэтому для разметки для каждой клетки с фигурой в ней рисовался BB размером 70x70. Эти данные сохранялись в формате .txt в формате для модели YOLO.

скриншот создания датасета в Blender

![image.png](../docs/assets/synthetic_annotations_example.png)

Пример размеченной фотографии

![image.png](../docs/assets/yolo_board_detection_real.png)

#### Освещение и выбор материала для фигур

Всего источников освещения было 5: в центре, и 4 на каждый угол доски.

Хорошее освещение помогло избавиться от теней у фигур, облегчая задачу модели CV. Но была ещё одна проблема: матовые фигуры невозможно было разглядеть на доске даже с хорошим освящением.

Чтобы решить эту проблему был задан материал Metallic с максимальным коэффициентом. Это сделало фигуры глянцевыми, отражения и блики от фигур сделали геометрию более понятной с исследуемого ракурса.

![0.png](../docs/assets/yolo_board_detection_real.png)

#### Общее количество фотографий

Всего фотографий искусственного датасета было сделано 15 000.

На каждой фотографии по 12 фигур - 180 000 размеченных фигур

### 2.1.2 Код

#### создание синтетического датасета в Blender

In [ ]:
import bpy
import random
import math
import os
import numpy as np

num_samples = 10000

model_dir = "C:/Program Files (x86)/Steam/steamapps/common/Blender/Project"
board_file = os.path.join(model_dir, "chess_board.obj")
piece_files = {
    "bishop_black": "bishop_black.obj",
    "bishop_white": "bishop_white.obj",
    "king_black": "king_black.obj",
    "king_white": "king_white.obj",
    "knight_black": "knight_black.obj",
    "knight_white": "knight_white.obj",
    "pawn_black": "pawn_black.obj",
    "pawn_white": "pawn_white.obj",
    "queen_black": "queen_black.obj",
    "queen_white": "queen_white.obj",
    "rook_black": "rook_black.obj",
    "rook_white": "rook_white.obj",
}

output_dir = "C:/Users/jack_/MAIN/ВУЗ/Курсовая/Dataset/Ver1"
os.makedirs(output_dir, exist_ok=True)
target_file = os.path.join(output_dir, "targets.csv")

# Очистка сцены
bpy.ops.object.select_all(action='SELECT')
bpy.ops.object.delete()

# Импортируем доску
bpy.ops.wm.obj_import(filepath=board_file)
board = bpy.context.selected_objects[0]
board.name = "Chessboard"

# Список всех фигур
pieces = {}

for piece_name, file_name in piece_files.items():
    bpy.ops.wm.obj_import(filepath=os.path.join(model_dir, file_name))
    piece = bpy.context.selected_objects[0]
    piece.name = piece_name
    piece.location = (100, 100, 0)  # Уводим за пределы видимости
    pieces[piece_name] = piece

bpy.ops.object.camera_add(location=(0, 0, 35))
cam = bpy.context.object
cam.rotation_euler = (0, 0, 0)
cam.data.type = 'PERSP'
bpy.context.scene.camera = cam

bpy.context.scene.render.resolution_x = 500
bpy.context.scene.render.resolution_y = 500
bpy.context.scene.render.resolution_percentage = 100


# Добавляем источник света
bpy.ops.object.light_add(type='SUN', location=(0, 0, 8))
light = bpy.context.object
light.rotation_euler = (0, 0, 0)

bpy.ops.object.light_add(type='AREA', location=(0, 0, 10))
bpy.context.object.data.energy = 1000
bpy.ops.object.light_add(type='AREA', location=(-10, -10, 10))
bpy.context.object.data.energy = 1000
bpy.ops.object.light_add(type='AREA', location=(+10, -10, 10))
bpy.context.object.data.energy = 1000
bpy.ops.object.light_add(type='AREA', location=(+10, +10, 10))
bpy.context.object.data.energy = 1000
bpy.ops.object.light_add(type='AREA', location=(-10, +10, 10))
bpy.context.object.data.energy = 1000


# Включаем рендеринг текстур и освещения
bpy.context.scene.render.engine = 'CYCLES'
#bpy.context.scene.render.engine = 'BLENDER_EEVEE'
bpy.context.scene.render.image_settings.file_format = 'PNG'
prefs = bpy.context.preferences.addons['cycles'].preferences
prefs.compute_device_type = 'CUDA'  # или 'OPTIX' / 'HIP' / 'METAL' — зависит от видеокарты
bpy.context.scene.cycles.device = 'GPU'
bpy.context.scene.cycles.samples = 32

# Получаем начальный номер итерации
start_id = 0
if os.path.exists(target_file):
    with open(target_file, "r") as f:
        lines = f.readlines()
        if len(lines) > 1:
            last_line = lines[-1]
            last_id = int(last_line.split(",")[0])
            start_id = last_id + 1

# Открываем файл в режиме дозаписи
with open(target_file, "a" if start_id > 0 else "w") as f:
    if start_id == 0:
        f.write("image_id,piece,x,y,rot_z\n")

    for i in range(start_id, start_id + num_samples):
        placed_pieces = []
        occupied_cells = set()

        for piece_name, piece in pieces.items():
            while True:
                place_x = np.random.randint(0, 8)
                place_y = np.random.randint(0, 8)
                if (place_x, place_y) not in occupied_cells:
                    occupied_cells.add((place_x, place_y))
                    break

            x = -8.96654 + place_x * 2.5474
            y = -8.91536 + place_y * 2.5474
            rot_z = random.uniform(0, 2 * math.pi)

            piece.location = (x, y, 1)
            piece.rotation_euler = (math.pi/2, 0, rot_z)

            placed_pieces.append((piece_name, place_x, place_y, rot_z))

        image_path = os.path.join(output_dir, f"{i}.png")
        bpy.context.scene.render.filepath = image_path
        bpy.ops.render.render(write_still=True)

        for piece_data in placed_pieces:
            f.write(f"{i},{piece_data[0]},{piece_data[1]},{piece_data[2]},{piece_data[3]}\n")

#### Разметка синтетического датасета

In [ ]:
data_path = r"D:\ВУЗ\Курсовая работа 2025\Dataset\ver3"
out_path = r"D:\ВУЗ\Курсовая работа 2025\Код + Dataset ver3"
csv_path = os.path.join(data_path, "targets.csv")
output_path = os.path.join(out_path, "datasets")
test_percent = 0.1

# Параметры доски
offset = 40            # отступ от края до доски
cell_size = 52         # размер клетки
img_size = 500         # размер изображения
board_size = 8

# Параметры bounding box
bb_w = 70              # ширина bb
bb_h = 70              # высота bb

# Чтение targets.csv
df = pd.read_csv(csv_path, header=None, names=["file", "label", "x", "y", "angle"], skiprows=1)
for i in ["x", "y"]:
    df[i] = df[i].apply(lambda k: np.int64(k))

# Список классов
labels = sorted(df["label"].unique())
label2id = {label: i for i, label in enumerate(labels)}

# Создание папок
for split in ["train", "test"]:
    os.makedirs(os.path.join(output_path, "images", split), exist_ok=True)
    os.makedirs(os.path.join(output_path, "labels", split), exist_ok=True)

# Разделение на train/test
all_files = sorted({f"{row}.png" for row in df["file"]})
random.shuffle(all_files)
split_idx = int(len(all_files) * (1 - test_percent))
train_files = set(all_files[:split_idx])
test_files = set(all_files[split_idx:])

# Генерация аннотаций
grouped = df.groupby("file")
for file_name, group in grouped:
    txt_lines = []
    for _, row in group.iterrows():
        cx = offset + (row["x"] + 0.5) * cell_size
        cy = img_size - (offset + (row["y"] + 0.5) * cell_size)
        yolo_x = cx / img_size
        yolo_y = cy / img_size
        yolo_w = bb_w / img_size
        yolo_h = bb_h / img_size
        class_id = label2id[row["label"]]
        txt_lines.append(f"{class_id} {yolo_x:.6f} {yolo_y:.6f} {yolo_w:.6f} {yolo_h:.6f}")

    split = "train" if f"{file_name}.png" in train_files else "test"
    with open(os.path.join(output_path, "labels", split, f"{file_name}.txt"), "w") as f:
        f.write("\n".join(txt_lines))

    shutil.copy(
        os.path.join(data_path, f"{file_name}.png"),
        os.path.join(output_path, "images", split, f"{file_name}.png")
    )

# Классы
with open(os.path.join(output_path, "classes.txt"), "w") as f:
    for label in labels:
        f.write(f"{label}\n")

## 2.2 Реальные фотографии

### 2.2.1 Описание

#### Зачем была сделана реальная часть датасета после создания синтетической

После пробных запусков модели YOLO, обученной только на синтетических данных, стало очевидным, что обобщающая способность модели для реальных данных низкая - сильно теряла точность от любых помех, таких как:
* Смещенный ракурс
* Потертость доски
* Неровно расположенные в рамках клетки фигуры
* Отличная от синтетического изображения цветовая гамма
* Неравномерное освещение
* Неравномерный не монотонный задний фон

и многое другое.



Пример некорректоного опредления BB и классификации модели, обученной только на синтетических данных:

![photo_5301178043991193853_x.jpg](../docs/assets/yolo_board_detection_real.png)

![photo_5301178043991193855_x.jpg](../docs/assets/yolo_board_detection_real.png)

#### Какие фигуры были использованы при создании, и как они были получены

Фигуры реального датасета - распечатанные на смолянном 3D принтере 3D модели синтетического датасета, позже покрашенные вручную.

#### Краска фигур и причина её выбора

Каждая фигура помимо соответствующего ей цвета была покрыта глянцевой краской. Это связано с тем, что матовые фигуры плохо различимы на доске даже при хорошем освещении невооруженным взглядом. Отражения и блики от фигур сделали геометрию более понятной с исследуемого ракурса.

до нанесения глянцевой краски:

![photo_5301178043991193270_y.jpg](../docs/assets/real_printed_figures_metallic.jpg)

после нанесения глянцевой краски:

![pawn_black_0008_white_angle_0.png](../docs/assets/real_single_piece_sample.png)

#### Условия создания датасета

Для создания датасета использовался прожектор, камера с возможностью снимать в разрешении 720p 30FPS, вращающийся стенд, скульптурный пластелин для создания угла наклона и картонная бумага с нарисованным клетками в качестве доски.

Для каждой фигуры было снято видео в 30 FPS в течении всего оборота

Один оборот составлял 24 секунды.

Видео для каждой фигуры было сделано на фоне черной и белой клетки отдельно, с тремя углами наклона: ~30 *, ~20 *, ~10 *.

Каждая фигура была помещена так, чтобы центр массы фигуры находился на линии перпендикуляра из центра вращающегося стенда. Это обеспечило несмещаемость фигуры относительно центра изображения на видео при повороте доски. Несмещаемость была необходима для автоматического выделения BB - ровно в центре видеоизображения.

Программа для записи была написана в Python: видео сразу снималось с рамками конечного изображения 500x500 с физуально размеченным BB в центре размером 70x70. Сразу после записи видео разрезалось на отдельные кадры, создавались подписи .txt в формате для YOLO и распределялись в отдельные папки для ручной проверки корректности работы программы.

##### Формирование тренировочного сета:

![Пояснения.png](../docs/assets/physical_capture_rig_schematic.png)

![rook_white_0104_black_angle_15.png](../docs/assets/yolo_board_detection_real.png)

##### Формирование тестового сета:

![photo_5301178043991193271_y.jpg](../docs/assets/yolo_board_detection_real.png)

![knight_black_0292_white_angle_0.png](../docs/assets/yolo_board_detection_real.png)

#### Количество фотографий

В итоге в реальном тренировочном датасете было 50 000 фотографий:

\

30 (кадров в секунду) * 24 (секунд на один оборот) = 720 фотографий за один оборот

720 * 2 = 1440 фотографий на фоне черной и белой клетки отдельно

1440 * 12 = 17 280 фотографий для каждой из 12 фигур

17 280 * 3 = 51 840 фотографий для каждого из 3 видов наклонов.

Итого: 51 840 фотографий.

\

Часть реального датасета была отведена на тестовые данные модели.

\

В итоге реальная часть датасета была получена за 3 часа.

### 2.2.2 Код виедосъемки

In [ ]:
cap = cv2.VideoCapture(1)
hw = 35
max_frames = 720
i = 0
record = False
piece = 'pawn'
colour = 'black'
checkerboard = 'white'
angle = '0'
name = f'{piece}_{colour}'


fig_names = '''bishop_black
bishop_white
king_black
king_white
knight_black
knight_white
pawn_black
pawn_white
queen_black
queen_white
rook_black
rook_white'''.split()
d = {fig_names[i]:i for i in range(len(fig_names))}


# Папки
root = f'TEST_real_set/{name}'
img_dir = os.path.join(root, 'images')
lbl_dir = os.path.join(root, 'labels')
os.makedirs(img_dir, exist_ok=True)
os.makedirs(lbl_dir, exist_ok=True)

print("Нажми 's' для старта записи, 'q' для выхода")

while True:
    ret, frame = cap.read()
    if not ret: break

    h, w, _ = frame.shape
    size = min(h, w)
    cx, cy = w//2, h//2
    x1, y1 = cx - size//2, cy - size//2
    x2, y2 = cx + size//2, cy + size//2

    crop = frame[y1:y2, x1:x2]  # центрированный квадрат
    f = cv2.resize(crop, (500, 500))

    bb_cx, bb_cy = 0.5, 0.5
    bb_w = bb_h = 70 / 500

    # превью
    p = f.copy()
    r1, r2 = 250 - hw, 250 + hw
    cv2.rectangle(p, (r1, r1), (r2, r2), (0,0,255), 1)
    cv2.imshow('setup', p)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('s'):
        print("Запись начата...")
        record = True
    if key == ord('q'):
        break

    if record and i < max_frames:
        fn = f'{name}_{i:04d}_{checkerboard}_angle_{angle}'
        cv2.imwrite(f'{img_dir}/{fn}.png', f)
        with open(f'{lbl_dir}/{fn}.txt', 'w') as L:
            L.write(f"{d[name]} {bb_cx} {bb_cy} {bb_w:.6f} {bb_h:.6f}")
        i += 1
        if i == max_frames:
            print("Максимум кадров достигнут.")
            break

cap.release()
cv2.destroyAllWindows()

Проверка камеры: изображение 500x500 с квадратом 70x70 ровно по центру

![image.png](../docs/assets/yolo_board_detection_real.png)

## 2.3 Проверка разметки

### код для проверки правильности разметки BB (сопоставление .txt и .png из labels и images)

In [ ]:

split = "train"
base_path = r"D:\ВУЗ\Курсовая работа 2025\Код + Dataset ver3\datasets"
img_dir = os.path.join(base_path, "images", split)
label_dir = os.path.join(base_path, "labels", split)
classes_path = os.path.join(base_path, "classes.txt")

with open(classes_path, "r") as f:
    id2label = [line.strip() for line in f]

img_name = random.choice(os.listdir(img_dir))
while img_name[-4:] != '.png':
    img_name = random.choice(os.listdir(img_dir))
img_path = os.path.join(img_dir, img_name)
txt_path = os.path.join(label_dir, img_name.replace(".png", ".txt"))

# Загрузка изображения и аннотации
def imread_unicode(path):
    stream = np.fromfile(path, dtype=np.uint8)
    return cv2.imdecode(stream, cv2.IMREAD_COLOR)
img = imread_unicode(img_path)
print(img_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
h, w, _ = img.shape

# Чтение аннотаций и отрисовка
if os.path.exists(txt_path):
    with open(txt_path, "r") as f:
        for line in f:
            cls, x, y, bw, bh = map(float, line.strip().split())
            x1 = int((x - bw / 2) * w)
            y1 = int((y - bh / 2) * h)
            x2 = int((x + bw / 2) * w)
            y2 = int((y + bh / 2) * h)
            label = id2label[int(cls)]

            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(img, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)
for i in range(12):
    print(i, id2label[i])

plt.figure(figsize=(6, 6))
plt.imshow(img)
plt.title(img_name)
plt.axis("off")
plt.show()

# 3. Обучения и тестирование моделей

## 3.1 Модель YOLO v11n

### 3.1.1 Введение

#### Главные особенности YOLO

YOLO - нейронная сеть, решающая задачу CV детекции путем выделения BB и классификации изображения. YOLO использует одноэтапный подход к детекции, что позволяет её работать в реальном времени. YOLO является SOTA (state of the art) моделью по детекции в реальном времени.

#### Различные модели YOLO, какая была выбрана для работы

YOLOv11 есть 5 видов в зависимости от размера весов:
* n - Nano
* s - Small
* m - Medium
* l - Large
* x - ExtraLarge

Для данной работы выбрана 11-ая версия YOLO размера Nano.

Маленький размер модели не повлиял значительно на качество обучения. Размер Nano облегчил проведение экспериментов из-за быстрой скорости сходимости.

#### Какие проводились запуски

* Пробная итерация - модель обучалась исключительно на синтетических данных без аугментаций.
* Первая итерация - обучение на синтетических данных без аугментации с повышенным коэффициентом ошибки Box_Loss (ошибка на разметку BB)
* Вторая итерация - обучение на аугментированных синтетических данных.
* Третья итерация - обучение на аугментированном конечном датасете, содержащим реальные и синтетические данные

#### Пояснения параметров аугментаций

hsv_h                # Изменение оттенка (Hue)

hsv_s                # Изменение насыщенности (Saturation)

hsv_v                # Изменение яркости (Value)



degrees                # Поворот изображения

translate            # Смещение изображения по x и y

scale                # Увеличение/уменьшение изображения

shear                 # Сдвиг (shear transformation)

perspective         # Искажение перспективы

flipud               # Вероятность отражения по вертикали

fliplr               # Вероятность отражения по горизонтали



mosaic               # склейка 4 изображений в одно

mixup                 # перемешивание изображений и аннотаций с весами



### Пояснение функции ошибки YOLO

Функция ошибки YOLO включает в себя 3 компоненты:

* box_loss - штраф ща неверное определение координат BB

* dfl_loss - тоже штраф за определение BB, но работает с вероятностями (предсказание вероятности нахождения BB)

* cls_loss - штраф за неверную классификацию



Формулы компонент функции ошибки:

\

$L_{box} = 1 - IoU + \frac{\delta + \omega}{2}$

$IoU$ - Intersection over Union - истинный BB накалдывается на реальный и считается отношение пересечения к объединению

$\delta$ - distance cost - чем ближе BB к истинному, тем меньше $\delta$

$\omega$ - shape cost - считает ошибку несовпадения соотношения сторон BB

\

* $L_{dfl}(S_i, S_j) = -((y_{j+1}-y)log(S_i) + (y-y_i)log(S_j))$

$S_i, S_j - \text{предсказанные вероятности для двух соседних дискретных значений соседних координат}$

$y - \text{истинное значение координаты в дискретизированном виде}$

$y_i<=y<y_{j+1} - интервал, в который попадает истинная координата$

\

$
\begin{cases}
-\alpha(1-p)^γlog(p)\text{, если y=1} \\
-(1-\alpha)p^γlog(1-p)\text{ иначе.}
\end{cases}
$

$\alpha$ - весовой коэффициент

$\gamma$ - параметр фокусировки

$p$ - вероятность правильного класса

$y$ - реальный класс (бинарный - 1 верный, 0 неверный класс)


$L_{YOLO} = L_{box} \cdot \text{box_loss} + L_{dfl} \cdot \text{dfl_loss} +L_{cls} \cdot \text{cls_loss}$

### 3.1.2 Пробная модель без предварительной обработки исключительно на синетических данных

#### Гиперпараметры обучения

Одна эпоха, все веса ф. ошибки по 1, без аугментаций

Пример входных синтетических данных

![val_batch1_labels.jpg](../docs/assets/yolo_board_detection_real.png)

#### Ход обучения

Модель обучилась на одной эпохе, показав идеальную точность на синтетических данных.

In [ ]:
pd.read_csv('results1.csv')

,epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2
0,1,147.267,0.61095,1.46991,0.91901,0.89903,0.89853,0.82168,0.32502,1.7298,0.75214,2.00163,0.000208,0.000208,0.000208


#### Результат

##### Тестирование на реальных данных

![image (1).png](../docs/assets/yolo_board_detection_real.png)

##### Тестирование на отложенных синтетиских данных

* box_loss: 0.61
* cls_loss: 1.469
* dfl_loss: 0.919

![F1_curve.png](../docs/assets/yolo_board_detection_real.png)![confusion_matrix.png](../docs/assets/yolo_board_detection_real.png)![PR_curve.png](../docs/assets/yolo_board_detection_real.png)

На этом примере видно, что модель крайне плохо предсказывает на реальных данных - неверная разметка и классификация.

#### Выводы

Модель идеально обучилась на входных данных, но не смогла делать предсказания на реальных тестовых фотографиях.

### 3.1.3 Модель на синтетических данных с модифицированными весам функции ошибки

#### Гиперпараметры обучения

##### Коэффициенты ф. ошибки

В самой первой итерации для box_loss был выбран 7 (остальные 1), потому что на тестовых данных bbox определялись неверно, и очень часто размечался на случайных местах доски.

box: 7.5 # Вес bbox-регрессии (насколько важны точные координаты)

cls: 1.0 # Вес классификационной ошибки (важность правильного определения класса)

dfl: 1.0 # Distribution Focal Loss (точность предсказания центра и размера bbox)



##### Аугментации

hsv_h: 0.0

hsv_s: 0.0

hsv_v: 0.0

degrees: 0.0

translate: 0.0

scale: 0.0

shear: 0.0

perspective: 0.0

flipud: 0.0

fliplr: 0.0

bgr: 0.0

mosaic: 0.0

mixup: 0.0

##### Пример тренировочного изображения

![val_batch0_labels.jpg](../docs/assets/yolo_board_detection_real.png)

#### Ход обучения

In [ ]:
pd.read_csv('results1_5.csv')

,epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2
0,1,123.507,0.77455,1.79787,1.02791,0.88597,0.88672,0.79716,0.29304,1.81739,0.71486,1.97087,0.000208,0.000208,0.000208


#### Результат

##### Результаты на реальной фотографии

![photo_5301178043991193853_x.jpg](../docs/assets/yolo_board_detection_real.png)

##### Результаты на отложенной части датасета

![val_batch2_labels.jpg](../docs/assets/yolo_board_detection_real.png)

![PR_curve.png](../docs/assets/yolo_board_detection_real.png)![F1_curve.png](../docs/assets/yolo_board_detection_real.png)![confusion_matrix.png](../docs/assets/yolo_board_detection_real.png)

#### Выводы

Повышение коэффициента ошибки Box_Loss не привел к ожидаемому результату. Обобщающая способность модели низкая без аугментации.

\

Из этого опыта был сделан вывод, что нужно добавить в датасет реальных данных (модель научится выделять фигуры из реального фона), добавить аугментации (чтобы минимальные искажения на тестовом датасете не влияли на модель, например освещение или угол поворота доски), повысить вес ошибки определения класса.

\

Но для начала было решено проэкспериментировать - просто добавить аугментации и посмотреть, сможет ли это достаточно увеличить обобщаемость модели, чтобы сделать правильные предсказания на реальных данных. \

### 3.1.4 Модель на аугментированных синтетических данных

#### Гиперпараметры обучения

##### Коэффициенты ф. ошибки

box: 1.5 # Вес bbox-регрессии (насколько важны точные координаты)

cls: 7.5 # Вес классификационной ошибки (важность правильного определения класса)

dfl: 1.0 # Distribution Focal Loss (точность предсказания центра и размера bbox)



##### Аугментации

Самой важной аугментацией здесь является mosaic - склеивание 4 случайных фотографий в одну (по 1/4 на каждый угол). Для YOLO в особенности это считается лучшей аугментацией ( скачок в точности YOLOv4 ещё в 2020 году из-за применения mosaic), потому что это делает фотографию насыщенной объектами - модель учится предсказывать много объектов за раз. Также это повышает обобщаемость, потому что объекты на mosaic-аугментированных изображениях мельче, а задний фон не совпадает с тем, что есть на монотонном датасете.

hsv_h: 0.015

hsv_s: 0.01

hsv_v: 0.01

degrees: 3.0

translate: 0.05

scale: 0.05

shear: 0.1

perspective: 0.001

flipud: 0.0

fliplr: 0.3

bgr: 0.0

mosaic: 1.0

mixup: 0.2

##### Пример аугментаций

![train_batch0.jpg](../docs/assets/yolo_board_detection_real.png)

#### Ход обучения

In [ ]:
pd.read_csv('results2.csv')

,epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2
0,1,77.647,0.25405,27.38740,0.86853,0.84499,0.84274,0.74474,0.27033,0.39279,11.06410,1.27630,0.003327,0.003327,0.003327
1,2,153.609,0.15481,10.96060,0.68178,0.81458,0.81471,0.68437,0.24322,0.41840,12.99440,1.46178,0.006594,0.006594,0.006594
2,3,282.567,0.14855,10.07480,0.66992,0.81311,0.79500,0.78174,0.30342,0.34880,29.23310,1.32318,0.009795,0.009795,0.009795
3,4,422.078,0.14449,9.34808,0.66312,0.71959,0.69353,0.67688,0.25587,0.39757,40.20580,1.57969,0.009703,0.009703,0.009703
4,5,493.015,0.13812,8.54752,0.65435,0.81859,0.81796,0.70730,0.25946,0.41775,15.12900,1.55851,0.009604,0.009604,0.009604
5,6,563.325,0.13426,8.11235,0.64817,0.88160,0.87978,0.80532,0.30888,0.36904,9.82953,1.39444,0.009505,0.009505,0.009505
6,7,632.400,0.13136,7.85179,0.64538,0.84143,0.84162,0.75000,0.29576,0.39607,10.47650,1.49245,0.009406,0.009406,0.009406
7,8,701.354,0.12950,7.67437,0.64311,0.86705,0.86735,0.80012,0.32937,0.38658,9.81911,1.50019,0.009307,0.009307,0.009307
8,9,771.011,0.12816,7.50907,0.64025,0.86561,0.86570,0.77410,0.29523,0.38234,9.94309,1.48625,0.009208,0.009208,0.009208
9,10,842.107,0.12627,7.37627,0.63922,0.87567,0.87590,0.81843,0.34862,0.37547,9.81782,1.45263,0.009109,0.009109,0.009109


![results.png](../docs/assets/yolo_board_detection_real.png)

#### Результат

##### Результаты реальной фотографии

![image.png](../docs/assets/yolo_board_detection_real.png)

##### Результаты на отложенной части датасета

![val_batch1_pred.jpg](../docs/assets/yolo_board_detection_real.png)

![PR_curve.png](../docs/assets/yolo_board_detection_real.png)![confusion_matrix.png](../docs/assets/yolo_board_detection_real.png)![F1_curve.png](../docs/assets/yolo_board_detection_real.png)

#### Выводы

Просто аугментации на синтетических данных не решили проблему обобщаемости предсказаний модели. Этот опыт доказал, что аугментации не могут научить модель игнорировать помехи на реальных данных (освещение, задний фон, цвет и т.д.)

### 3.1.5 Модель на полном датасете, включающим реальные и ситнтетические аугментированные данные

#### Гиперпараметры обучения

##### Коэффициенты ф. ошибки

Box_loss я повысил до 1.5, а классификацию выставил 5.0:
Т.к. 3/4 (180 000 к 50 000 + 180 000) размеченных фигур датасета искусственные, они дают модели хорошее понимание как размечать на фотографии BB, а оставшаяся четверть даёт модели хорошее представление о геометрии фигур, но на них BB находится ровно в центре. Для того, чтобы выровнять этот дисбаланс определения геометрии, и потому что определить сами фигуры на доске бывает и невооруженным глазом сложно, вес ошибки классификации должен был стать большим.



box: 1.5                    # Вес bbox-регрессии (насколько важны точные координаты)

cls: 5.0                    # Вес классификационной ошибки (важность правильного определения класса)

dfl: 1.0                    # Distribution Focal Loss (точность предсказания центра и размера bbox)


##### Аугментации

Для аугментаций было внесено несколько изменений :

* повышено случайное изменение яркости, чтобы модель лучше обобщалась к разным источникам света

* увеличено амплитуду случайного увеличения размера фигур - при небольших наклонах камеры для тестового датасета модель должна давать точные результаты



hsv_h: 0.015                # Изменение оттенка (Hue) - занизил значения, негативно влияло на обучение

hsv_s: 0.01                 # Изменение насыщенности (Saturation)

hsv_v: 0.01                 # Изменение яркости (Value)



degrees: 3.0                # Поворот изображения

translate: 0.05            # Смещение изображения по x и y

scale: 0.05                # Увеличение/уменьшение изображения

shear: 0.1                 # Сдвиг (shear transformation)

perspective: 0.001         # Искажение перспективы

flipud: 0.0                # Вероятность отражения по вертикали

fliplr: 0.3                # Вероятность отражения по горизонтали



mosaic: 1.0               # склейка 4 изображений в одно — оказалась очень полезной, при включении точность увеличилась

mixup: 0.2                 # перемешивание изображений и аннотаций с весами



##### Пример аугментаций

![train_batch0.jpg](../docs/assets/yolo_mosaic_augmentation.jpg)

#### Ход обучения

In [ ]:
pd.read_csv('results3.csv')

,epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2
0,1,199.863,0.25121,15.80390,0.77989,0.84501,0.84525,0.74541,0.26954,0.37113,8.02930,1.03756,0.003331,0.003331,0.003331
1,2,402.673,0.17400,8.04007,0.66353,0.86649,0.86809,0.77166,0.29825,0.35163,7.16166,1.02290,0.006599,0.006599,0.006599
2,3,599.348,0.16913,7.27227,0.65852,0.84885,0.84620,0.78766,0.31152,0.34113,7.87273,1.01148,0.009800,0.009800,0.009800
3,4,794.096,0.16603,6.76767,0.65468,0.87406,0.87414,0.79630,0.33258,0.30411,5.81006,0.97867,0.009703,0.009703,0.009703
4,5,988.884,0.15849,6.22381,0.64595,0.88008,0.88031,0.81469,0.35719,0.29834,5.48890,0.98056,0.009604,0.009604,0.009604
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69,70,13541.700,0.11083,4.14523,0.59187,0.89050,0.89059,0.85355,0.40742,0.23486,4.36970,0.91025,0.003169,0.003169,0.003169
70,71,13733.800,0.10986,4.09694,0.59017,0.89013,0.89021,0.85310,0.40696,0.23487,4.37303,0.91089,0.003070,0.003070,0.003070
71,72,13926.600,0.11042,4.11628,0.59088,0.89021,0.89029,0.85305,0.40749,0.23479,4.37856,0.91100,0.002971,0.002971,0.002971
72,73,14120.300,0.11021,4.10398,0.59049,0.89004,0.89012,0.85290,0.40721,0.23481,4.38183,0.91165,0.002872,0.002872,0.002872


![results.png](../docs/assets/yolo_training_curves_74epochs.png)

#### Результат

Модель обучалась 3ч 40мин: 2:45 мин на эпоху, 75 эпох

##### Результаты на реальных фотграфиях

![final2.png](../docs/assets/yolo_board_detection_real.png)

##### Результаты на отложенной части датасета

![val_batch0_pred.jpg](../docs/assets/yolo_val_batch_predictions.jpg)

![confusion_matrix.png](../docs/assets/yolo_confusion_matrix.png)![PR_curve.png](../docs/assets/yolo_confusion_matrix.png)![F1_curve.png](../docs/assets/yolo_confusion_matrix.png)

#### Выводы

Модель почти идеально научилась распозновать фигуры на реальных тестовых данных. Ошибки модель делает в двух местах: кода фигура находится с краю изображения и когда фигура черная и не бликует.

Это лушая модель YOLO, полученная в данной работе.

Комбинация идеально размеченных синтетических данных с разнообразными координатами bbox и реальных фигур под разными углами оказалась успешной. В случае маленькой точности планировалось добавить в датасет размеченные руками фотографии реальной доски со всеми фигурами на ней.

### 3.1.6 Код обучения моделей YOLOv11n и вывода тестовых данных

#### Обучение

In [ ]:
if torch.cuda.is_available():
    device = 'cuda'
    torch.cuda.set_device(0)
    print(f"CUDA is available. Using device: {torch.cuda.get_device_name(0)}")
else:
    device = 'cpu'
    print("CUDA is not available. Using CPU.")

model = YOLO("yolo11n.pt").to(device)
print(f"Model device: {model.device}")


results = None
results = model.train(cfg="Ultralytics.yaml")

if results:
    metrics = model.val()
    print(f"mAP50-95: {metrics.box.map}")


if results:
    model.save("model_ver_3.pt")
    print("Model saved as model_ver_3.pt")

#### Вывод изображений

In [ ]:
model_path = r"D:\ВУЗ\Курсовая работа 2025\Код + Dataset ver3\runs\detect\yolov11n_chess3\weights\best.pt"
model = YOLO(model_path)

def plot_img(path):
    img_path = path

    results = model.predict(source=img_path, save=False, conf=0.25)

    res_plotted = results[0].plot()  # numpy-массив с отрисованными bb
    img = Image.fromarray(res_plotted)
    plt.imshow(img)
    plt.axis("off")
    plt.show()

imgs =[
    r"D:\ВУЗ\Курсовая работа 2025\Код + Dataset ver2\Final test\try1.png",
    r"D:\ВУЗ\Курсовая работа 2025\Код + Dataset ver2\Final test\try2.png",
    r"D:\ВУЗ\Курсовая работа 2025\Код + Dataset ver2\Final test\test1.png",
    r"D:\ВУЗ\Курсовая работа 2025\Код + Dataset ver2\Final test\test2.png",
    r"D:\ВУЗ\Курсовая работа 2025\Код + Dataset ver2\Final test\test3.png",
]

for i in imgs:
    plot_img(i)


image 1/1 D:\\  2025\ + Dataset ver2\Final test\try1.png: 512x512 1 bishop_black, 1 bishop_white, 2 king_whites, 2 knight_blacks, 1 knight_white, 1 pawn_white, 1 queen_black, 2 queen_whites, 2 rook_blacks, 1 rook_white, 2.8ms
Speed: 0.8ms preprocess, 2.8ms inference, 1.2ms postprocess per image at shape (1, 3, 512, 512)


<Figure size 640x480 with 1 Axes>


image 1/1 D:\\  2025\ + Dataset ver2\Final test\try2.png: 512x512 2 bishop_blacks, 2 bishop_whites, 1 king_black, 1 king_white, 6 knight_blacks, 1 knight_white, 1 pawn_black, 2 pawn_whites, 1 queen_black, 2 queen_whites, 1 rook_black, 2 rook_whites, 3.0ms
Speed: 0.9ms preprocess, 3.0ms inference, 0.7ms postprocess per image at shape (1, 3, 512, 512)


<Figure size 640x480 with 1 Axes>


image 1/1 D:\\  2025\ + Dataset ver2\Final test\test1.png: 512x512 2 king_blacks, 2 knight_blacks, 1 pawn_black, 2 queen_blacks, 4.2ms
Speed: 1.1ms preprocess, 4.2ms inference, 0.7ms postprocess per image at shape (1, 3, 512, 512)


<Figure size 640x480 with 1 Axes>


image 1/1 D:\\  2025\ + Dataset ver2\Final test\test2.png: 512x512 1 knight_black, 1 pawn_black, 2 queen_blacks, 1 rook_black, 4.6ms
Speed: 0.9ms preprocess, 4.6ms inference, 0.8ms postprocess per image at shape (1, 3, 512, 512)


<Figure size 640x480 with 1 Axes>


image 1/1 D:\\  2025\ + Dataset ver2\Final test\test3.png: 512x512 2 king_blacks, 3 knight_blacks, 2 knight_whites, 1 pawn_black, 4 queen_blacks, 3 queen_whites, 1 rook_white, 3.3ms
Speed: 0.9ms preprocess, 3.3ms inference, 0.8ms postprocess per image at shape (1, 3, 512, 512)


<Figure size 640x480 with 1 Axes>

In [ ]:
model_path = r"D:\ВУЗ\Курсовая работа 2025\Код + Dataset ver3\runs\detect\yolov11n_chess2\weights\best.pt"
model = YOLO(model_path)

def plot_img(path):
    img_path = path

    results = model.predict(source=img_path, save=False, conf=0.25)

    res_plotted = results[0].plot()  # numpy-массив с отрисованными bb
    img = Image.fromarray(res_plotted)
    plt.imshow(img)
    plt.axis("off")
    plt.show()

imgs = [f'D:\ВУЗ\Курсовая работа 2025\Код + Dataset ver3\TEST_real_set\\bishop_white\images\\bishop_white_{str(i).zfill(4)}_white_angle_0.png' for i in range(0,720,100)]

for i in imgs:
    print(i)
    plot_img(i)

D:\ВУЗ\Курсовая работа 2025\Код + Dataset ver3\TEST_real_set\bishop_white\images\bishop_white_0000_white_angle_0.png



<>:15: SyntaxWarning: invalid escape sequence '\T'
<>:15: SyntaxWarning: invalid escape sequence '\T'
C:\Users\kv\AppData\Local\Temp\ipykernel_23672\2089758949.py:15: SyntaxWarning: invalid escape sequence '\T'
  imgs = [f'D:\ВУЗ\Курсовая работа 2025\Код + Dataset ver3\TEST_real_set\\bishop_white\images\\bishop_white_{str(i).zfill(4)}_white_angle_0.png' for i in range(0,720,100)]


image 1/1 D:\\  2025\ + Dataset ver3\TEST_real_set\bishop_white\images\bishop_white_0000_white_angle_0.png: 512x512 1 bishop_black, 1 bishop_white, 1 pawn_black, 1 pawn_white, 1 queen_black, 1 queen_white, 86.5ms
Speed: 2.3ms preprocess, 86.5ms inference, 1.4ms postprocess per image at shape (1, 3, 512, 512)


<Figure size 640x480 with 1 Axes>

D:\ВУЗ\Курсовая работа 2025\Код + Dataset ver3\TEST_real_set\bishop_white\images\bishop_white_0100_white_angle_0.png

image 1/1 D:\\  2025\ + Dataset ver3\TEST_real_set\bishop_white\images\bishop_white_0100_white_angle_0.png: 512x512 1 bishop_black, 1 bishop_white, 1 pawn_black, 1 pawn_white, 1 queen_black, 1 queen_white, 3.1ms
Speed: 0.9ms preprocess, 3.1ms inference, 0.7ms postprocess per image at shape (1, 3, 512, 512)


<Figure size 640x480 with 1 Axes>

... (8 outputs left out)

D:\ВУЗ\Курсовая работа 2025\Код + Dataset ver3\TEST_real_set\bishop_white\images\bishop_white_0600_white_angle_0.png

image 1/1 D:\\  2025\ + Dataset ver3\TEST_real_set\bishop_white\images\bishop_white_0600_white_angle_0.png: 512x512 1 bishop_black, 1 bishop_white, 1 pawn_black, 1 pawn_white, 1 queen_black, 1 queen_white, 3.6ms
Speed: 0.9ms preprocess, 3.6ms inference, 0.8ms postprocess per image at shape (1, 3, 512, 512)


<Figure size 640x480 with 1 Axes>

D:\ВУЗ\Курсовая работа 2025\Код + Dataset ver3\TEST_real_set\bishop_white\images\bishop_white_0700_white_angle_0.png

image 1/1 D:\\  2025\ + Dataset ver3\TEST_real_set\bishop_white\images\bishop_white_0700_white_angle_0.png: 512x512 1 bishop_black, 1 bishop_white, 1 pawn_black, 1 pawn_white, 1 queen_black, 1 queen_white, 3.2ms
Speed: 0.9ms preprocess, 3.2ms inference, 0.7ms postprocess per image at shape (1, 3, 512, 512)


<Figure size 640x480 with 1 Axes>

In [ ]:
model_path = r"D:\ВУЗ\Курсовая работа 2025\Код + Dataset ver3\runs\detect\yolov11n_chess2\weights\best.pt"
model = YOLO(model_path)
def plot_img(path):
    img_path = path

    results = model.predict(source=img_path, save=False, conf=0.25)

    res_plotted = results[0].plot()  # numpy-массив с отрисованными bb
    img = Image.fromarray(res_plotted)
    plt.imshow(img)
    plt.axis("off")
    plt.show()

imgs = [f'D:\ВУЗ\Курсовая работа 2025\Код + Dataset ver3\TEST_real_set\king_white\images\king_white_{str(i).zfill(4)}_white_angle_0.png' for i in range(0,720,100)]

for i in imgs:
    print(i)
    plot_img(i)

D:\ВУЗ\Курсовая работа 2025\Код + Dataset ver3\TEST_real_set\king_white\images\king_white_0000_white_angle_0.png

image 1/1 D:\\  2025\ + Dataset ver3\TEST_real_set\king_white\images\king_white_0000_white_angle_0.png: 512x512 1 bishop_black, 1 knight_black, 1 knight_white, 1 queen_white, 9.9ms
Speed: 1.3ms preprocess, 9.9ms inference, 5.1ms postprocess per image at shape (1, 3, 512, 512)


<>:14: SyntaxWarning: invalid escape sequence '\T'
<>:14: SyntaxWarning: invalid escape sequence '\T'
C:\Users\kv\AppData\Local\Temp\ipykernel_23672\2994376725.py:14: SyntaxWarning: invalid escape sequence '\T'
  imgs = [f'D:\ВУЗ\Курсовая работа 2025\Код + Dataset ver3\TEST_real_set\king_white\images\king_white_{str(i).zfill(4)}_white_angle_0.png' for i in range(0,720,100)]


<Figure size 640x480 with 1 Axes>

D:\ВУЗ\Курсовая работа 2025\Код + Dataset ver3\TEST_real_set\king_white\images\king_white_0100_white_angle_0.png

image 1/1 D:\\  2025\ + Dataset ver3\TEST_real_set\king_white\images\king_white_0100_white_angle_0.png: 512x512 1 bishop_black, 1 knight_white, 1 queen_white, 13.7ms
Speed: 1.3ms preprocess, 13.7ms inference, 1.3ms postprocess per image at shape (1, 3, 512, 512)


<Figure size 640x480 with 1 Axes>

D:\ВУЗ\Курсовая работа 2025\Код + Dataset ver3\TEST_real_set\king_white\images\king_white_0200_white_angle_0.png

image 1/1 D:\\  2025\ + Dataset ver3\TEST_real_set\king_white\images\king_white_0200_white_angle_0.png: 512x512 1 bishop_black, 1 knight_white, 1 queen_white, 11.5ms
Speed: 1.4ms preprocess, 11.5ms inference, 2.4ms postprocess per image at shape (1, 3, 512, 512)


... (7 outputs left out)

D:\ВУЗ\Курсовая работа 2025\Код + Dataset ver3\TEST_real_set\king_white\images\king_white_0600_white_angle_0.png

image 1/1 D:\\  2025\ + Dataset ver3\TEST_real_set\king_white\images\king_white_0600_white_angle_0.png: 512x512 1 bishop_black, 1 knight_black, 1 knight_white, 1 queen_white, 8.4ms
Speed: 1.4ms preprocess, 8.4ms inference, 1.2ms postprocess per image at shape (1, 3, 512, 512)


<Figure size 640x480 with 1 Axes>

D:\ВУЗ\Курсовая работа 2025\Код + Dataset ver3\TEST_real_set\king_white\images\king_white_0700_white_angle_0.png

image 1/1 D:\\  2025\ + Dataset ver3\TEST_real_set\king_white\images\king_white_0700_white_angle_0.png: 512x512 1 bishop_black, 1 knight_black, 1 knight_white, 1 queen_white, 8.9ms
Speed: 1.3ms preprocess, 8.9ms inference, 1.6ms postprocess per image at shape (1, 3, 512, 512)


<Figure size 640x480 with 1 Axes>

In [ ]:
model_path = r"D:\ВУЗ\Курсовая работа 2025\Код + Dataset ver3\runs\detect\yolov11n_chess2\weights\best.pt"
model = YOLO(model_path)
def plot_img(path):
    img_path = path

    results = model.predict(source=img_path, save=False, conf=0.25)

    res_plotted = results[0].plot()  # numpy-массив с отрисованными bb
    img = Image.fromarray(res_plotted)
    plt.imshow(img)
    plt.axis("off")
    plt.show()

imgs = [f'D:\ВУЗ\Курсовая работа 2025\Код + Dataset ver3\TEST_real_set\pawn_black\images\pawn_black_{str(i).zfill(4)}_white_angle_0.png' for i in range(0,720,100)]

for i in imgs:
    print(i)
    plot_img(i)

D:\ВУЗ\Курсовая работа 2025\Код + Dataset ver3\TEST_real_set\pawn_black\images\pawn_black_0000_white_angle_0.png



<>:14: SyntaxWarning: invalid escape sequence '\T'
<>:14: SyntaxWarning: invalid escape sequence '\T'
C:\Users\kv\AppData\Local\Temp\ipykernel_23672\2787139596.py:14: SyntaxWarning: invalid escape sequence '\T'
  imgs = [f'D:\ВУЗ\Курсовая работа 2025\Код + Dataset ver3\TEST_real_set\pawn_black\images\pawn_black_{str(i).zfill(4)}_white_angle_0.png' for i in range(0,720,100)]


image 1/1 D:\\  2025\ + Dataset ver3\TEST_real_set\pawn_black\images\pawn_black_0000_white_angle_0.png: 512x512 1 bishop_white, 1 king_black, 1 king_white, 1 knight_white, 1 pawn_black, 1 pawn_white, 1 queen_black, 1 queen_white, 1 rook_white, 2.9ms
Speed: 0.8ms preprocess, 2.9ms inference, 0.7ms postprocess per image at shape (1, 3, 512, 512)


<Figure size 640x480 with 1 Axes>

D:\ВУЗ\Курсовая работа 2025\Код + Dataset ver3\TEST_real_set\pawn_black\images\pawn_black_0100_white_angle_0.png

image 1/1 D:\\  2025\ + Dataset ver3\TEST_real_set\pawn_black\images\pawn_black_0100_white_angle_0.png: 512x512 1 bishop_white, 1 king_black, 1 knight_white, 1 pawn_black, 1 pawn_white, 2 queen_blacks, 2 queen_whites, 1 rook_black, 1 rook_white, 3.2ms
Speed: 0.9ms preprocess, 3.2ms inference, 0.8ms postprocess per image at shape (1, 3, 512, 512)


<Figure size 640x480 with 1 Axes>

... (8 outputs left out)

D:\ВУЗ\Курсовая работа 2025\Код + Dataset ver3\TEST_real_set\pawn_black\images\pawn_black_0600_white_angle_0.png

image 1/1 D:\\  2025\ + Dataset ver3\TEST_real_set\pawn_black\images\pawn_black_0600_white_angle_0.png: 512x512 1 bishop_white, 1 king_black, 1 king_white, 1 knight_white, 1 queen_black, 1 queen_white, 1 rook_white, 3.4ms
Speed: 1.0ms preprocess, 3.4ms inference, 0.7ms postprocess per image at shape (1, 3, 512, 512)


<Figure size 640x480 with 1 Axes>

D:\ВУЗ\Курсовая работа 2025\Код + Dataset ver3\TEST_real_set\pawn_black\images\pawn_black_0700_white_angle_0.png

image 1/1 D:\\  2025\ + Dataset ver3\TEST_real_set\pawn_black\images\pawn_black_0700_white_angle_0.png: 512x512 1 bishop_black, 1 bishop_white, 1 king_black, 1 knight_white, 1 pawn_black, 1 pawn_white, 1 queen_black, 1 queen_white, 1 rook_white, 3.3ms
Speed: 0.9ms preprocess, 3.3ms inference, 0.7ms postprocess per image at shape (1, 3, 512, 512)


<Figure size 640x480 with 1 Axes>

## 3.2 Модель SSD

Для решения данной задачи CV также была выбрана SSD (Google Research, 2015). Это полностью сверточная нейронная сеть. Как YOLO, SSD может работать в реальном времени. Одной из ключевых особенностей SSD является оптимизрованное применение anchor boxes (впервые появившиеся в Faster RCNN) - жестко заданных регионов-кандидатов для feature maps.

Для backbone был выбран VGG16.

Скорость обучения SSD ниже YOLO11n: 17 минут на эпоху вместо трёх. Конечная модель заняла 4 ч 25 минут для обучения.

Конечная версия SSD обучалась на 15 эпохах с батчем 8.

На этапе обучения пришлось использовать только Adam в качестве оптимизатора - SGD давал взрывающиеся градиенты на первой же эпохе, и снижение скорости обучения лишь в некторых случаях позволяло модели застрять в локальном минимуме и не уйти в бесконечность.

Были добавлены аугментации данных в связи с низкой точностью пробных первых вариантов SSD. Значния для аугментаций были взяты из значений для YOLO.

### Код обучения и проверки модели

#### Аугментации

In [ ]:
# параметры из YAML
hsv_h, hsv_s, hsv_v = 0.015, 0.01, 0.01
degrees     = 3.0
translate   = 0.05
scale       = 0.05
shear       = 0.1
perspective = 0.001
flipud, fliplr = 0.0, 0.3

# аугментации
w = VGG16_Weights.DEFAULT
mean, std = w.transforms().mean, w.transforms().std

train_aug = A.Compose([
    A.Resize(300, 300),
    A.HueSaturationValue(
        hue_shift_limit=int(hsv_h * 360),
        sat_shift_limit=int(hsv_s * 255),
        val_shift_limit=int(hsv_v * 255),
        p=1.0
    ),
    A.Affine(
        rotate=degrees,
        translate_percent=translate,
        scale=(1 - scale, 1 + scale),
        shear=shear,
        p=1.0
    ),
    A.Perspective(
        scale=(0, perspective),
        keep_size=True,
        p=perspective > 0
    ),
    A.HorizontalFlip(p=fliplr),
    A.VerticalFlip(p=flipud),
    A.Normalize(mean=mean, std=std),
    ToTensorV2()
], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))

#### Создание представления датасета для SSD с выводом аугментированных фотографий

Переходя от YOLO к SSD и Faster RCNN пришлось поменять разметку: перейти от [x_center, y_center, width, height] к [x_min, y_min, x_max, y_max] для BBox.

In [ ]:
class DS(Dataset):
    def __init__(self, img_dir, lbl_dir, cache='ds_ssd_improved.pt', rebuild=False):
        self.img_dir = img_dir;
        w = VGG16_Weights.DEFAULT
        self.transforms = T.Compose([
            T.Resize((300,300)),
            T.ToTensor(),
            T.Normalize(mean=w.transforms().mean, std=w.transforms().std)
        ])
        self.aug = train_aug
        if os.path.exists(cache) and not rebuild:
            self.data = torch.load(cache)
        else:
            self.data = []
            for fn in os.listdir(img_dir):
                if not fn.endswith(('.jpg','.png')): continue
                img = Image.open(f"{img_dir}/{fn}").convert("RGB")
                w,h = img.size
                boxes, lbls = [], []
                for l in open(f"{lbl_dir}/{fn.rsplit('.',1)[0]}.txt"):
                    c, xc,yc,ww,hh = map(float,l.split())
                    x1=(xc-ww/2)*w; y1=(yc-hh/2)*h
                    x2=(xc+ww/2)*w; y2=(yc+hh/2)*h
                    boxes.append([x1,y1,x2,y2]); lbls.append(int(c) + 1)
                tgt={"boxes":torch.tensor(boxes),
                     "labels":torch.tensor(lbls)}
                self.data.append((fn, tgt))
            torch.save(self.data, cache)
        self.n = len(self.data)

    def __len__(self): return self.n

    def __getitem__(self, i):
        fn, tgt = self.data[i]
        img = np.array(Image.open(f"{self.img_dir}/{fn}").convert("RGB"))
        bbs, lbls = tgt['boxes'].tolist(), tgt['labels'].tolist()
        aug_out = self.aug(image=img, bboxes=bbs, labels=lbls)
        img_aug = aug_out['image']
        boxes = torch.tensor(aug_out['bboxes'], dtype=torch.float32)
        labels = torch.tensor(aug_out['labels'], dtype=torch.int64)

        return img_aug, {'boxes': boxes, 'labels': labels}


ds = DS('D:/ВУЗ/Курсовая работа 2025/Код + Dataset ver3/datasets/images/train','D:/ВУЗ/Курсовая работа 2025/Код + Dataset ver3/datasets/labels/train', rebuild=False)
dl = DataLoader(ds, batch_size=8, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))

#### Проверка, что созданное представление выводит правильную разметку

In [ ]:
i = random.randrange(len(ds))
img, tgt = ds[i]

boxes = tgt['boxes']
labels = tgt['labels']
print(f"Image #{i}, boxes:", boxes)


#print(img[:50,:50])

mean = np.array([0.485, 0.456, 0.406])
std  = np.array([0.229, 0.224, 0.225])
img_np = img.permute(1,2,0).cpu().numpy()
img_np = (img_np * std + mean) * 255
img_np = np.clip(img_np, 0, 255).astype(np.uint8)
img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
#print(img_bgr)
print(labels)

if boxes.numel() == 0:
    raise RuntimeError("Нет ни одного bbox")

for box, lbl in zip(boxes, labels):
    x1, y1, x2, y2 = box.round().long().tolist()
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(299, x2), min(299, y2)
    cv2.rectangle(img_bgr, (x1, y1), (x2, y2), (0,255,0), 2)
    cv2.putText(img_bgr, str(int(lbl)), (x1, y1-5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)

plt.figure(figsize=(5,5))
plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
plt.title(f"Sample {i}, {len(boxes)} boxes")
plt.axis('off')
plt.show()

Image #28884, boxes: tensor([[144.1072, 143.5895, 186.8261, 187.7357]])
tensor([5])


<Figure size 500x500 with 1 Axes>

#### Обучение SSD с 15 эпохами размером батча 8

In [ ]:
m = ssd300_vgg16(weights=None,num_classes=13,weights_backbone=VGG16_Weights.DEFAULT)

#opt = torch.optim.SGD(m.parameters(), lr=0.00005, momentum=0.9, weight_decay=0.0005)
opt = torch.optim.AdamW(m.parameters(), lr=1e-4, weight_decay=5e-4)

lr_s = torch.optim.lr_scheduler.StepLR(opt, step_size=3, gamma=0.1)
#lr_s = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, 'min',patience=3)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('cuda' if torch.cuda.is_available() else 'cpu')


m.to(device)

m.train()
print(f'Training started')
loss=0
for epoch in range(15):
    cycle = tqdm(dl, desc=f"Epoch {epoch}")
    for imgs, tgts in cycle:
        imgs = [i.to(device) for i in imgs]
        tgts = [{k: v.to(device) for k, v in t.items()} for t in tgts]

        loss_dict = m(imgs, tgts)
        loss = sum(loss_dict.values())
        #print(loss)
        cycle.set_postfix(loss=np.float64(loss))
        opt.zero_grad(); loss.backward(); opt.step()
    lr_s.step()
    print(f"Epoch {epoch} loss {loss:.4f}")
print(f'Training ended')
torch.save(m.state_dict(), 'SSD_IMPROVED_V2.pth')

#### Локальная проверка модели

In [ ]:
m = ssd300_vgg16(weights=None,num_classes=13,weights_backbone=VGG16_Weights.DEFAULT)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

m.load_state_dict(torch.load('SSD_IMPROVED_V2.pth', map_location=device))
m.to(device)

d = {i:j for i,j in zip((str(k) for k in range(13)),'''Background
bishop_black
bishop_white
king_black
king_white
knight_black
knight_white
pawn_black
pawn_white
queen_black
queen_white
rook_black
rook_white'''.split())}

def vis(img_path, model, cls_names=None, thresh=0.5):
    img = Image.open(img_path).convert("RGB")
    x = T.ToTensor()(img).to(device)
    model.eval()
    with torch.no_grad():
        out = model([x])[0]

    img_cv = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    for box, cls, score in zip(out['boxes'], out['labels'], out['scores']):
        #print(score)
        if score < thresh: continue
        x1,y1,x2,y2 = map(int, box)
        label_ = str(cls.item()) if not cls_names else cls_names[cls.item()]
        label = [d[i] for i in label_]
        cv2.rectangle(img_cv, (x1,y1), (x2,y2), (0,255,0), 2)
        cv2.putText(img_cv, f"{label} {score:.2f}", (x1,y1-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)
    plt.imshow(cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()

imgs = [f'D:\ВУЗ\Курсовая работа 2025\Код + Dataset ver3\TEST_real_set\pawn_black\images\pawn_black_{str(i).zfill(4)}_white_angle_0.png' for i in range(0,720,100)]
for i in imgs:
    vis(i,m,thresh=0.3)

<>:42: SyntaxWarning: invalid escape sequence '\T'
<>:42: SyntaxWarning: invalid escape sequence '\T'
C:\Users\kv\AppData\Local\Temp\ipykernel_23672\4260850814.py:42: SyntaxWarning: invalid escape sequence '\T'
  imgs = [f'D:\ВУЗ\Курсовая работа 2025\Код + Dataset ver3\TEST_real_set\pawn_black\images\pawn_black_{str(i).zfill(4)}_white_angle_0.png' for i in range(0,720,100)]


<Figure size 640x480 with 1 Axes>

<Figure size 640x480 with 1 Axes>

<Figure size 640x480 with 1 Axes>

<Figure size 640x480 with 1 Axes>

<Figure size 640x480 with 1 Axes>

<Figure size 640x480 with 1 Axes>

<Figure size 640x480 with 1 Axes>

<Figure size 640x480 with 1 Axes>

### Результаты

##### Результаты на реальных фотографиях

![predictions_custom3.png](../docs/assets/ssd_predictions_real1.png)

![predictions_custom6.png](../docs/assets/ssd_predictions_real2.png)

![predictions_custom1.png](../docs/assets/yolo_board_detection_real.png)

##### Результаты на отложенной части датасета

![predictions_test.png](../docs/assets/ssd_predictions_test.png)

![confusion_matrix.png](../docs/assets/ssd_confusion_matrix.png)

![pr_curve_per_class.png](../docs/assets/ssd_pr_curve.png)

### Вывод

Модель показывает адекватные обобщающие сопосбности в рамках задачи. По PR кривой можно заметить, что модель недообучена: по Confusion Matrix заметно, что модель не всегда находит фигуры на доске, но те, что находит, классифицирует верно.



## 3.3 Модель Faster RCNN

FasterRCNN это нейронная сеть, решающая задачу детекции CV используя двухстадийную архитектуру. Метод требует больших датасетов и длительного обучения, но это компенисируется высокой точностью.

Из-за низкой скорости сходимости количество картинок в batch пришлось уменьшить, но на качестве классификации это плохо не отразилось.

Для Backbone я выбрал ResNet50 (98 мб). Cлишком сложный backbone привел бы к переобучению на созданном мною датасете.

Конечная модель FasterRCNN была обучена за 8 ч 9 минут на 10 эпохпх с батчем 4.

### Код обучения и проверки модели

#### код для конвертации

In [ ]:
class DS(Dataset):
    def __init__(self, img_dir, lbl_dir, cache='ds.pt', rebuild=False):
        self.img_dir = img_dir;
        if os.path.exists(cache) and not rebuild:
            self.data = torch.load(cache)
        else:
            self.data = []
            for fn in os.listdir(img_dir):
                if not fn.endswith(('.jpg','.png')): continue
                img = Image.open(f"{img_dir}/{fn}").convert("RGB")
                w,h = img.size
                boxes, lbls = [], []
                for l in open(f"{lbl_dir}/{fn.rsplit('.',1)[0]}.txt"):
                    c, xc,yc,ww,hh = map(float,l.split())
                    x1=(xc-ww/2)*w; y1=(yc-hh/2)*h
                    x2=(xc+ww/2)*w; y2=(yc+hh/2)*h
                    boxes.append([x1,y1,x2,y2]); lbls.append(int(c) + 1)
                tgt={"boxes":torch.tensor(boxes),
                     "labels":torch.tensor(lbls)}
                self.data.append((fn, tgt))
            torch.save(self.data, cache)
        self.n = len(self.data)

    def __len__(self): return self.n
    def __getitem__(self, i):
        fn, tgt = self.data[i]
        img = Image.open(f"{self.img_dir}/{fn}").convert("RGB")
        return T.ToTensor()(img), tgt


ds = DS('D:/ВУЗ/Курсовая работа 2025/Код + Dataset ver3/datasets/images/train','D:/ВУЗ/Курсовая работа 2025/Код + Dataset ver3/datasets/labels/train', rebuild=False)
dl = DataLoader(ds, batch_size=4, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))

#### проверка правильности разметки

In [ ]:
i = random.randrange(len(ds))
img, tgt = ds[i]

boxes = tgt['boxes']
labels = tgt['labels']
print(f"Image #{i}, boxes:", boxes)

#print(img[:50,:50])

img_np = img.permute(1,2,0).cpu().numpy()

img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
#print(img_bgr)
print(labels)

if boxes.numel() == 0:
    raise RuntimeError("Нет ни одного bbox")

for box, lbl in zip(boxes, labels):
    x1, y1, x2, y2 = box.round().long().tolist()
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(np.array(img_bgr).shape[0], x2), min(np.array(img_bgr).shape[1], y2)
    cv2.rectangle(img_bgr, (x1, y1), (x2, y2), (0,255,0), 2)
    cv2.putText(img_bgr, str(int(lbl)), (x1, y1-5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)


plt.figure(figsize=(5,5))
plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
plt.title(f"Sample {i}, {len(boxes)} boxes")
plt.axis('off')
plt.show()

Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [0.0..255.0].


Image #8824, boxes: tensor([[135., 191., 205., 261.],
        [ 83., 139., 153., 209.],
        [ 31., 295., 101., 365.],
        [135., 347., 205., 417.],
        [343., 243., 413., 313.],
        [343., 399., 413., 469.],
        [343., 347., 413., 417.],
        [ 31., 139., 101., 209.],
        [239.,  87., 309., 157.],
        [395., 191., 465., 261.],
        [395., 139., 465., 209.],
        [291., 243., 361., 313.]])
tensor([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12])


<Figure size 500x500 with 1 Axes>

#### обучение модели

Вместо SGD был выбран ADAM в качестве оптимизатора из-за высокого прироста к скорости обучения.

In [ ]:
m = fasterrcnn_resnet50_fpn(weights=None, weights_backbone=ResNet50_Weights.DEFAULT,num_classes=13)  # С = число классов +1
#opt = torch.optim.SGD(m.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)
opt = torch.optim.AdamW(m.parameters(), lr=1e-4, weight_decay=5e-4)
lr_s = torch.optim.lr_scheduler.StepLR(opt, step_size=3, gamma=0.1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('cuda' if torch.cuda.is_available() else 'cpu')
m.to(device)

m.train()
print(f'Training started')
for epoch in range(10):
    cycle = tqdm(dl, desc=f"Epoch {epoch}")
    for imgs, tgts in cycle:
        imgs = [i.to(device) for i in imgs]
        tgts = [{k: v.to(device) for k, v in t.items()} for t in tgts]

        loss_dict = m(imgs, tgts)
        loss = sum(loss_dict.values())
        cycle.set_postfix(loss=np.float64(loss))

        opt.zero_grad(); loss.backward(); opt.step()
    lr_s.step()
    print(f"Epoch {epoch} loss {loss:.4f}")
print(f'Training ended')
torch.save(m.state_dict(), 'frcnn_IMPROVED_V2.pth')

#### проверка обученной модели

In [ ]:
m = fasterrcnn_resnet50_fpn(weights=None, weights_backbone=ResNet50_Weights.DEFAULT, num_classes=13)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

m.load_state_dict(torch.load('frcnn_IMPROVED_V2.pth', map_location=device))
m.to(device)

d = {i:j for i,j in zip((str(k) for k in range(13)),'''Background
bishop_black
bishop_white
king_black
king_white
knight_black
knight_white
pawn_black
pawn_white
queen_black
queen_white
rook_black
rook_white'''.split())}

w = ResNet50_Weights.DEFAULT
mean, std = w.transforms().mean, w.transforms().std
preproc = T.Compose([
    T.Resize((500,500)),
    T.ToTensor(),
    T.Normalize(mean=mean, std=std)
])

def vis(img_path, model, cls_names=None, thresh=0.5):
    img = Image.open(img_path).convert("RGB")
    x = T.ToTensor()(img).to(device)
    #x = preproc(img).to(device)
    model.eval()
    with torch.no_grad():
        out = model([x])[0]
    print( out['scores'])
    img_cv = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    for box, cls, score in zip(out['boxes'], out['labels'], out['scores']):
        if score < thresh: continue
        x1,y1,x2,y2 = map(int, box)
        label_ = str(cls.item()) if not cls_names else cls_names[cls.item()]
        label = [d[i] for i in label_]
        cv2.rectangle(img_cv, (x1,y1), (x2,y2), (0,255,0), 2)
        cv2.putText(img_cv, f"{label} {score:.2f}", (x1,y1-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)
    plt.imshow(cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()

imgs = [f'D:\ВУЗ\Курсовая работа 2025\Код + Dataset ver3\TEST_real_set\pawn_black\images\pawn_black_{str(i).zfill(4)}_white_angle_0.png' for i in range(0,720,100)]
for i in imgs:
    vis(i,m)

<>:50: SyntaxWarning: invalid escape sequence '\T'
<>:50: SyntaxWarning: invalid escape sequence '\T'
C:\Users\kv\AppData\Local\Temp\ipykernel_23672\947961934.py:50: SyntaxWarning: invalid escape sequence '\T'
  imgs = [f'D:\ВУЗ\Курсовая работа 2025\Код + Dataset ver3\TEST_real_set\pawn_black\images\pawn_black_{str(i).zfill(4)}_white_angle_0.png' for i in range(0,720,100)]


tensor([1.0000, 1.0000, 1.0000, 0.9999, 0.9999, 0.9999, 0.9990], device='cuda:0')


<Figure size 640x480 with 1 Axes>

tensor([1.0000, 0.9999, 0.9999, 0.9995, 0.9971, 0.9705, 0.9404, 0.9082, 0.7298, 0.6589, 0.3814, 0.1776, 0.0514], device='cuda:0')


<Figure size 640x480 with 1 Axes>

tensor([1.0000, 1.0000, 0.9999, 0.9989, 0.9971, 0.9935, 0.3182, 0.1986], device='cuda:0')


... (7 outputs left out)

tensor([1.0000, 0.9999, 0.9999, 0.9997, 0.9933, 0.9494, 0.9186, 0.5856, 0.4127, 0.4071, 0.1717, 0.0854], device='cuda:0')


<Figure size 640x480 with 1 Axes>

tensor([0.9999, 0.9994, 0.9992, 0.9901, 0.5887, 0.4984, 0.2629, 0.2376, 0.0985, 0.0644], device='cuda:0')


<Figure size 640x480 with 1 Axes>

### результат

##### Результаты на реальных фотографиях

![predictions_custom3.png](../docs/assets/faster_rcnn_predictions_real1.png)![predictions_custom2.png](../docs/assets/faster_rcnn_predictions_real1.png)

![predictions_custom5.png](../docs/assets/faster_rcnn_predictions_real3.png)

##### Результаты на отложенной части датасета

![confusion_matrix.png](../docs/assets/faster_rcnn_confusion_matrix.png)

![pr_curve_per_class.png](../docs/assets/faster_rcnn_pr_curve.png)

### вывод

Модель имеет адекватную обобщающую способность в рамках поставленной задачи, что видно на предсказаниях на реальных фотографиях, но переобучилась на входных данных. \

Вывод о переобучении модели был сделан по тому, как несколько эпох подряд ошибка модели не изменялась, и по тому, что модель обучалась на не аугментированных данных (на аугментированных данных модель не сходилась за 8ч - ошибка была постоянной и не уменьшалась со временем). Этот недостаток заметно снизил обобщающую способность модели.

Переобучение связано с недостаточным объемом и разнообразием данных датасета для FasterRCNN.


# 4. Выводы

Моделью с самый высокой точностью выделения и детекции фигур из полученных оказалась YOLOv11n. Она продемонстрировала лучшую обобщающую способность на реальных тренировочных данных затратив на обучение наименьшее количество времени.

Модель SSD, несмотря на недообученность, давала вторые по качеству предсказания на реальных данных после YOLO. Недообучение связано с плохой сходимостью из-за жесткости anchor boxes. Сходимость модели была достигнута заменой и оптимизатора с SGD на Adam, подбором скорости обучения и уменьшением размера батча.

Модели FasterRCNN требовался датасет большего размера с более разнообразными изображениями - реальными фотографиями с разным расположением фигур на доске. Несмотря на это модель в большинстве случаев верно определяет фигуры. Сходимость модели была достигнута путем уменьшения размера батча. Скорость сходимости была увеличина заменой SGD на Adam.

Оптимизатор Adam показал более стабильную работу и более высокую по сравнению с SGD скорость работы.

# 5. Литература и источники информации

[Формат разметки BB](https://lohithmunakala.medium.com/bounding-box-formats-for-models-like-yolo-ssd-rcnn-fast-rcnn-faster-rcnn-807be7721527)

[официальный краткий разбор модели YOLOv11](https://docs.ultralytics.com/ru/models/yolo11/)

[Формула ошибки YOLO](https://learnopencv.com/yolo-loss-function-siou-focal-loss/)

[Официальная документация FasterRCNN](https://docs.pytorch.org/vision/main/models/generated/torchvision.models.detection.fasterrcnn_resnet50_fpn.html)

[Главная страница PyTorch для SSD](https://pytorch.org/hub/nvidia_deeplearningexamples_ssd/)

[Методы оптимизации в Deep Learning](https://education.yandex.ru/handbook/ml/article/metody-optimizacii-v-deep-learning)



\

Официальные научные статьи обученных в рамках работы моделей:

[YOLOV11: AN OVERVIEW OF THE KEY ARCHITECTURAL
ENHANCEMENTS](https://arxiv.org/pdf/2410.17725v1)

[Faster R-CNN: Towards Real-Time Object
Detection with Region Proposal Networks](https://arxiv.org/pdf/1506.01497)

[SSD: Single Shot MultiBox Detector](https://arxiv.org/pdf/1512.02325)


# 0. Imports

In [ ]:
import os
import shutil
import random
import time
from tqdm import tqdm

import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt

from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision.models.detection import fasterrcnn_resnet50_fpn, ssd300_vgg16
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights
from torchvision.models import ResNet50_Weights, VGG16_Weights
from torchvision.models.detection.backbone_utils import resnet_fpn_backbone

from ultralytics import YOLO

import albumentations as A
from albumentations.pytorch import ToTensorV2